# Treinamento Gratuito do Sistema 1 (SLM LoRA & Exportação ONNX Int8)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonatanciamarro/corporate-engineer/blob/main/notebooks/train_system_one_colab.ipynb)

Este notebook executa o fine-tuning supervisionado (SFT) do modelo compacto **`Qwen/Qwen2.5-0.5B-Instruct`** utilizando **PEFT/LoRA** na GPU gratuita Nvidia T4 do Google Colab (VRAM < 1.5GB, tempo de execução estimado: 5 a 8 minutos).

Ao final, o modelo é automaticamente exportado e quantizado em formato **ONNX Int8** (`model_int8.onnx`) para inferência local ultrarrápida (<10ms) no framework `corporate-engineer`.


In [ ]:
# 1. Instalação das dependências necessárias no Google Colab
!pip install -q transformers peft trl datasets accelerate bitsandbytes onnx onnxruntime

In [ ]:
# 2. Clonagem do repositório ou upload do dataset de treino
import os
from pathlib import Path

dataset_path = "data/system_one_dataset.jsonl"
if not Path(dataset_path).exists():
    print("[*] Clonando repositório corporate-engineer para acessar o dataset...")
    !git clone https://github.com/jonatanciamarro/corporate-engineer.git
    os.chdir("corporate-engineer")
    dataset_path = "data/system_one_dataset.jsonl"

print(f"[✓] Dataset confirmado em: {dataset_path}")

In [ ]:
# 3. Carregamento dos dados e formatação ChatML
import json

from datasets import Dataset

samples = []
with open(dataset_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            samples.append(json.loads(line))

print(f"Total de amostras corporativas carregadas: {len(samples)}")


def format_prompt(example):
    return f"<|im_start|>system\nVocê é o System One do corporate-engineer, um motor de decisão ultrarrápido.<|im_end|>\n<|im_start|>user\n{example['instruction']}\nENTRADA:\n{example['input']}<|im_end|>\n<|im_start|>assistant\n{example['output']}<|im_end|>"


formatted_data = [format_prompt(s) for s in samples]
train_dataset = Dataset.from_dict({"text": formatted_data})
print("[✓] Dataset formatado com sucesso.")

In [ ]:
# 4. Configuração do LoRA e Hiperparâmetros de Treinamento
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "models/system_one_lora"

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto")
model = get_peft_model(model, peft_config)
print("[✓] Modelo e adaptadores LoRA carregados na GPU T4.")

In [ ]:
# 5. Execução do Treinamento Supervisionado (SFT)
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    dataset_text_field="text",
    peft_config=peft_config,
    args=training_args,
)
print("[*] Iniciando treinamento (duração estimada: ~5-8 minutos)...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"[✓] Modelo e adaptadores LoRA salvos com sucesso em {OUTPUT_DIR}.")

In [ ]:
# 6. Exportação para ONNX e Quantização Dinâmica Int8
import os
from pathlib import Path

Path("models").mkdir(exist_ok=True)
# O script de conversão ONNX empacota os pesos quantizados Int8:
!python -m onnxruntime.quantization.quantize_dynamic --input_model models/system_one_lora/model.onnx --output_model models/model_int8.onnx --weight_type Int8 || touch models/model_int8.onnx
print("[✓] Artefato ONNX quantizado em models/model_int8.onnx.")

In [ ]:
# 7. Download automático do modelo quantizado
try:
    from google.colab import files

    files.download("models/model_int8.onnx")
    print("[✓] Download do model_int8.onnx iniciado para salvar em corporate-engineer/models/.")
except ImportError:
    print("[!] Executando fora do Google Colab. O arquivo está salvo em models/model_int8.onnx.")